# Translon Consensus Talk Foundation

Clean, inspectable dataframes and simple talk-ready plots for the translon-consensus pilot. The notebook deliberately separates dataframe construction from plotting so each plot can be tweaked quickly tomorrow morning.

Main rules:
- query the translon DB once into named tables;
- calculate agreement in genomic space;
- add transcript/candidate context afterward;
- use P-site-registered bigWig scores before interpreting periodicity;
- keep plots simple and slide-readable.


## 1. Setup and safety checks

This cell prints the exact scorer module being used. If `Scorer supports P-site offsets` is false, sync the updated `transcode_bigwig_signal_scores.py` before trusting periodicity plots.


In [ ]:
import inspect
import json
import sqlite3
import sys
from pathlib import Path

import matplotlib
try:
    from IPython import get_ipython
    _ip = get_ipython()
    if _ip is None:
        matplotlib.use('Agg')
    else:
        _ip.run_line_magic('matplotlib', 'inline')
except Exception:
    matplotlib.use('Agg')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SCRIPT_DIR = Path.cwd()
if (SCRIPT_DIR / 'transcode_bigwig_signal_scores.py').exists():
    sys.path.insert(0, str(SCRIPT_DIR))
else:
    sys.path.insert(0, str(Path('/Users/jackt/projects/all-RiboSeq/ensembl-genes-nf/pipelines/translon-consensus/scripts')))

import transcode_bigwig_signal_scores as signal_scores

ScoreConfig = signal_scores.ScoreConfig
score_database = signal_scores.score_database
expand_scores_by_tool = signal_scores.expand_scores_by_tool
discover_bigwigs = signal_scores.discover_bigwigs
HAS_PYBIGWIG = signal_scores.HAS_PYBIGWIG

if hasattr(signal_scores, 'load_psite_offsets'):
    load_psite_offsets = signal_scores.load_psite_offsets
else:
    def load_psite_offsets(path):
        path = Path(path)
        if not path.exists():
            print(f'P-site offsets file {path} is absent; using offset 0 for all samples')
            return {}
        raw = json.loads(path.read_text())
        offsets = {}
        for sample_id, value in raw.items():
            offset = value.get('offset') if isinstance(value, dict) else value
            offsets[str(sample_id)] = int(offset) % 3
        return offsets

try:
    from calibrate_psite_offsets import calibrate_sample, load_mane_cds
    HAS_CALIBRATION_HELPERS = True
except Exception as exc:
    print(f'Calibration helpers unavailable: {exc}')
    HAS_CALIBRATION_HELPERS = False

DB = Path('/hps/nobackup/flicek/ensembl/genebuild/jackt/riboseq/pilot/full_pilot_results/translon_db_rebuild/translon_db/translons.sqlite')
GTF = Path('/hps/nobackup/flicek/ensembl/genebuild/jackt/riboseq/pilot/full_pilot_results/gencode.v47.annotation.gtf')
GROUPED_BIGWIG_ROOT = Path('/hps/nobackup/flicek/ensembl/genebuild/jackt/riboseq/gencode_pilot_pancreas/grouped_pilot_tracks/bigwigs')

OUT_DIR = Path('figures_talk_foundation')
TABLE_DIR = Path('tables_talk_foundation')
OUT_DIR.mkdir(exist_ok=True)
TABLE_DIR.mkdir(exist_ok=True)

SIGNAL_MODE = 'unique_no_junction'
BIGWIG_ROOT = OUT_DIR / f'bigwig_links_{SIGNAL_MODE}'
BIGWIG_ROOT.mkdir(parents=True, exist_ok=True)
PSITE_OFFSETS = OUT_DIR / 'psite_offsets.json'
SCORES_PATH = OUT_DIR / 'translon_signal_scores.tsv.gz'
SUMMARY_PATH = OUT_DIR / 'signal_score_summary.json'

PANCREAS_SAMPLES = [
    'SRR11005875_to_79',
    'SRR11005880_to_84',
    'SRR11005885_to_89',
    'SRR11005890_to_94',
    'SRR11005895_to_99',
    'SRR11005900_to_04',
    'Ribo_Pancreas_pooled',
]
PRIMARY_SAMPLE = 'Ribo_Pancreas_pooled'
TOOL_ORDER = ['PRICE', 'RiboTIE', 'ORFQuant', 'iRibo', 'RibORF2']
AGREEMENT_ORDER = ['full_consensus', 'boundary_agreement', 'tool_specific']
CONTEXT_ORDER = ['annotated_CDS', 'ambiguous_mixed_class', 'non_CDS_AUG', 'non_CDS_near_cognate', 'non_CDS_other_start']
TOOL_COLORS = {
    'PRICE': '#4C72B0',
    'RiboTIE': '#DD8452',
    'ORFQuant': '#55A868',
    'iRibo': '#C44E52',
    'RibORF2': '#8172B2',
}
CONTEXT_COLORS = {
    'annotated_CDS': '#4C72B0',
    'ambiguous_mixed_class': '#8172B2',
    'non_CDS_AUG': '#55A868',
    'non_CDS_near_cognate': '#DD8452',
    'non_CDS_other_start': '#C44E52',
}
AGREEMENT_COLORS = {
    'full_consensus': '#4C72B0',
    'boundary_agreement': '#DD8452',
    'tool_specific': '#55A868',
}

SCORER_MODULE = Path(signal_scores.__file__).resolve()
SCORER_SUPPORTS_PSITE = (
    hasattr(signal_scores, 'load_psite_offsets')
    and 'psite_offsets' in inspect.signature(score_database).parameters
    and 'psite_offset' in getattr(ScoreConfig, '__dataclass_fields__', {})
)

print(f'DB: {DB} exists={DB.exists()}')
print(f'GTF: {GTF} exists={GTF.exists()}')
print(f'Grouped BigWig root: {GROUPED_BIGWIG_ROOT} exists={GROUPED_BIGWIG_ROOT.exists()}')
print(f'Output dir: {OUT_DIR.resolve()}')
print(f'Table dir: {TABLE_DIR.resolve()}')
print(f'Scorer module: {SCORER_MODULE}')
print(f'Scorer supports P-site offsets: {SCORER_SUPPORTS_PSITE}')
print(f'pyBigWig available: {HAS_PYBIGWIG}')
if not SCORER_SUPPORTS_PSITE:
    print('WARNING: sync transcode_bigwig_signal_scores.py before recalculating registered periodicity.')


## 2. Small helpers

These helpers keep the later cells short. They do not hide analysis decisions: each dataframe is still created explicitly in its own section.


In [ ]:
def show_df(name, df, n=5):
    print(f'{name}: shape={df.shape}')
    if len(df):
        display(df.head(n))
    else:
        print('(empty)')


def save_table(df, name):
    path = TABLE_DIR / f'{name}.tsv'
    df.to_csv(path, sep='\t', index=False)
    print(f'Saved {path}')
    return path


def finish_plot(fig, out_name):
    out = OUT_DIR / out_name
    fig.tight_layout()
    fig.savefig(out, dpi=180, bbox_inches='tight')
    plt.show()
    print(f'Saved {out}')
    return out


def genomic_start_stop(df):
    start = np.where(df['bed_strand'].eq('+'), df['bed_start'], df['bed_end'])
    stop = np.where(df['bed_strand'].eq('+'), df['bed_end'], df['bed_start'])
    return pd.Series(start, index=df.index).astype(int), pd.Series(stop, index=df.index).astype(int)


def add_genomic_keys(df):
    out = df.copy()
    out['genomic_start'], out['genomic_stop'] = genomic_start_stop(out)
    out['full_key'] = (
        out['bed_chrom'].astype(str) + ':' + out['bed_strand'].astype(str) + ':'
        + out['genomic_start'].astype(str) + ':' + out['genomic_stop'].astype(str) + ':'
        + out['block_sizes'].astype(str) + ':' + out['block_starts'].astype(str)
    )
    out['start_key'] = out['bed_chrom'].astype(str) + ':' + out['bed_strand'].astype(str) + ':' + out['genomic_start'].astype(str)
    out['stop_key'] = out['bed_chrom'].astype(str) + ':' + out['bed_strand'].astype(str) + ':' + out['genomic_stop'].astype(str)
    return out


def compact_support_tier(n):
    n = int(n)
    if n >= 4:
        return '4+ tools'
    return f'{n} tool' if n == 1 else f'{n} tools'


def ordered_present(values, order):
    present = set(values)
    return [x for x in order if x in present] + sorted(present - set(order))


def summary_has_matching_offsets(summary, offsets):
    samples = summary.get('samples', []) if isinstance(summary, dict) else []
    if not offsets or not samples:
        return False
    for sample in samples:
        sid = str(sample.get('sample_id'))
        if 'psite_offset' not in sample:
            return False
        if int(sample['psite_offset']) != int(offsets.get(sid, 0)):
            return False
    return True


## 3. Query the DB into `calls_raw`

`calls_raw` is the base table: one QC-pass DB row per candidate occurrence. The rest of the notebook derives smaller tables from this.


In [ ]:
BASE_FILTER = """
    qc_status = 'pass'
    AND sample_id NOT GLOB '*_fastq'
    AND source_feature_class IN ('cds', 'non_cds')
"""

if not DB.exists():
    print(f'DB is unavailable: {DB}')
    calls_raw = pd.DataFrame(columns=[
        'source_tool', 'sample_id', 'cls', 'feature_key', 'bed_chrom', 'bed_start', 'bed_end',
        'bed_strand', 'block_sizes', 'block_starts', 'spliced_length_nt', 'start_codon_class',
        'terminal_codon_class'
    ])
else:
    with sqlite3.connect(DB) as con:
        calls_raw = pd.read_sql_query(f"""
            SELECT source_tool,
                   sample_id,
                   source_feature_class AS cls,
                   feature_key,
                   bed_chrom,
                   bed_start,
                   bed_end,
                   bed_strand,
                   block_sizes,
                   block_starts,
                   spliced_length_nt,
                   start_codon_class,
                   terminal_codon_class
            FROM translons
            WHERE {BASE_FILTER}
        """, con)

calls_raw['source_tool'] = calls_raw['source_tool'].astype(str).str.strip() if len(calls_raw) else calls_raw.get('source_tool', pd.Series(dtype=str))
show_df('calls_raw', calls_raw)
if len(calls_raw):
    display(calls_raw.groupby(['sample_id', 'source_tool', 'cls'])['feature_key'].nunique().reset_index(name='unique_features').head(20))


## 4. Pooled calls, candidates, agreement, and context

Agreement is calculated in genomic space. Candidate context is layered on afterward.


In [ ]:
calls_pooled = calls_raw[calls_raw['sample_id'].eq(PRIMARY_SAMPLE)].copy()
if calls_pooled.empty and len(calls_raw):
    print(f'Primary sample {PRIMARY_SAMPLE!r} not found. Falling back to all rows for agreement summaries.')
    calls_pooled = calls_raw.copy()

calls_pooled = (
    calls_pooled
    .sort_values(['source_tool', 'feature_key'])
    .drop_duplicates(['source_tool', 'feature_key'])
    .reset_index(drop=True)
)
calls_pooled = add_genomic_keys(calls_pooled) if len(calls_pooled) else calls_pooled
save_table(calls_pooled, 'calls_pooled')
show_df('calls_pooled', calls_pooled)


In [ ]:
if calls_pooled.empty:
    candidate_df = pd.DataFrame()
else:
    candidate_df = (
        calls_pooled
        .groupby('feature_key', sort=False)
        .agg(
            bed_chrom=('bed_chrom', 'first'),
            bed_start=('bed_start', 'first'),
            bed_end=('bed_end', 'first'),
            bed_strand=('bed_strand', 'first'),
            block_sizes=('block_sizes', 'first'),
            block_starts=('block_starts', 'first'),
            spliced_length_nt=('spliced_length_nt', 'first'),
            genomic_start=('genomic_start', 'first'),
            genomic_stop=('genomic_stop', 'first'),
            full_key=('full_key', 'first'),
            start_key=('start_key', 'first'),
            stop_key=('stop_key', 'first'),
            source_tools=('source_tool', lambda s: '|'.join(ordered_present(s.astype(str), TOOL_ORDER))),
            n_tools_full=('source_tool', 'nunique'),
        )
        .reset_index()
    )

save_table(candidate_df, 'candidate_df')
show_df('candidate_df', candidate_df)


In [ ]:
if calls_pooled.empty:
    agreement_df = pd.DataFrame()
else:
    start_counts = calls_pooled.groupby('start_key')['source_tool'].nunique().rename('n_tools_start')
    stop_counts = calls_pooled.groupby('stop_key')['source_tool'].nunique().rename('n_tools_stop')
    agreement_df = candidate_df.merge(start_counts, on='start_key', how='left').merge(stop_counts, on='stop_key', how='left')
    agreement_df[['n_tools_start', 'n_tools_stop']] = agreement_df[['n_tools_start', 'n_tools_stop']].fillna(0).astype(int)
    agreement_df['agreement_class'] = np.select(
        [
            agreement_df['n_tools_full'].ge(2),
            agreement_df['n_tools_full'].eq(1) & (agreement_df['n_tools_start'].ge(2) | agreement_df['n_tools_stop'].ge(2)),
        ],
        ['full_consensus', 'boundary_agreement'],
        default='tool_specific',
    )

save_table(agreement_df, 'agreement_df')
show_df('agreement_df', agreement_df)
if len(agreement_df):
    display(agreement_df['agreement_class'].value_counts().rename_axis('agreement_class').reset_index(name='candidates'))


In [ ]:
def classify_context(group):
    classes = set(group['cls'].dropna().astype(str))
    starts = set(group['start_codon_class'].dropna().astype(str))
    if {'cds', 'non_cds'}.issubset(classes):
        return 'ambiguous_mixed_class'
    if 'cds' in classes:
        return 'annotated_CDS'
    if 'ATG' in starts:
        return 'non_CDS_AUG'
    if 'near_cognate' in starts:
        return 'non_CDS_near_cognate'
    return 'non_CDS_other_start'

if calls_pooled.empty:
    context_df = pd.DataFrame()
else:
    context_df = (
        calls_pooled
        .groupby('feature_key', sort=False)
        .apply(lambda g: pd.Series({
            'context_class': classify_context(g),
            'classes': '|'.join(sorted(set(g['cls'].dropna().astype(str)))),
            'start_codon_classes': '|'.join(sorted(set(g['start_codon_class'].dropna().astype(str)))),
            'terminal_codon_classes': '|'.join(sorted(set(g['terminal_codon_class'].dropna().astype(str)))),
            'spliced_length_nt': g['spliced_length_nt'].min(),
        }))
        .reset_index()
    )

save_table(context_df, 'context_df')
show_df('context_df', context_df)
if len(context_df):
    display(context_df['context_class'].value_counts().rename_axis('context_class').reset_index(name='candidates'))


## 5. BigWig links, P-site offsets, and registered scores

The periodicity evidence plots must be skipped unless the score summary records the matching per-sample P-site offsets.


In [ ]:
# Create scorer-compatible symlinks: {sample}.forward.bw / {sample}.reverse.bw
link_rows = []
for sample in PANCREAS_SAMPLES:
    for strand in ('forward', 'reverse'):
        src = GROUPED_BIGWIG_ROOT / f'{sample}.{SIGNAL_MODE}.{strand}.bw'
        dst = BIGWIG_ROOT / f'{sample}.{strand}.bw'
        if dst.exists() or dst.is_symlink():
            dst.unlink()
        if src.exists():
            dst.symlink_to(src)
        link_rows.append({'sample_id': sample, 'strand': strand, 'source_path': src, 'link_path': dst, 'source_exists': src.exists()})

bigwig_links_df = pd.DataFrame(link_rows)
save_table(bigwig_links_df, 'bigwig_links')
show_df('bigwig_links_df', bigwig_links_df, 14)

bw_manifest = discover_bigwigs(BIGWIG_ROOT)
show_df('bw_manifest', bw_manifest)


In [ ]:
# Calibrate offsets if needed; otherwise load the existing frozen file.
if PSITE_OFFSETS.exists():
    offsets_json = json.loads(PSITE_OFFSETS.read_text())
    print(f'Loaded existing offsets: {PSITE_OFFSETS}')
elif not (HAS_PYBIGWIG and HAS_CALIBRATION_HELPERS and GTF.exists() and not bw_manifest.empty):
    offsets_json = {}
    print('Skipping calibration: pyBigWig, calibration helpers, GTF, or BigWigs are unavailable.')
else:
    transcripts = load_mane_cds(GTF)
    print(f'{len(transcripts):,} MANE_Select transcripts with CDS + start codon')
    offsets_json = {}
    for row in bw_manifest[bw_manifest['has_pair']].sort_values('sample_id').itertuples(index=False):
        if str(row.sample_id) not in PANCREAS_SAMPLES:
            continue
        offsets_json[str(row.sample_id)] = calibrate_sample(str(row.sample_id), str(row.fwd_path), str(row.rev_path), transcripts)
    PSITE_OFFSETS.write_text(json.dumps(offsets_json, indent=2) + '\n')
    print(f'Wrote {PSITE_OFFSETS}')

plot_psite_offsets_df = pd.DataFrame([{'sample_id': sid, **vals} for sid, vals in offsets_json.items()])
if not plot_psite_offsets_df.empty:
    plot_psite_offsets_df['offset_ok'] = plot_psite_offsets_df['offset'].eq(1)
    plot_psite_offsets_df['frame_ok'] = plot_psite_offsets_df['dom_frame_plus'].eq(2) & plot_psite_offsets_df['dom_frame_minus'].eq(2)
    plot_psite_offsets_df['strength_ok'] = plot_psite_offsets_df['dom_frac'].gt(0.8)

save_table(plot_psite_offsets_df, '05_psite_offsets')
show_df('plot_psite_offsets_df', plot_psite_offsets_df, 10)


In [ ]:
# Run registered scoring if possible; otherwise load any existing score table but mark it unregistered unless the summary proves otherwise.
psite_offsets = load_psite_offsets(PSITE_OFFSETS) if PSITE_OFFSETS.exists() else {}
cfg = ScoreConfig(
    flank_nt=30,
    body_edge_nt=15,
    min_body_nt=30,
    min_total_signal=10.0,
    max_features_per_sample=None,
    max_total_features=None,
    overwrite=True,
)

if HAS_PYBIGWIG and SCORER_SUPPORTS_PSITE and DB.exists() and BIGWIG_ROOT.exists() and psite_offsets:
    scores_unique, score_summary = score_database(DB, BIGWIG_ROOT, OUT_DIR, cfg, psite_offsets=psite_offsets)
else:
    if SCORES_PATH.exists():
        print(f'Loading existing scores: {SCORES_PATH}')
        scores_unique = pd.read_csv(SCORES_PATH, sep='\t')
    else:
        print('No scores available yet.')
        scores_unique = pd.DataFrame()
    score_summary = json.loads(SUMMARY_PATH.read_text()) if SUMMARY_PATH.exists() else {'samples': []}

SCORES_ARE_REGISTERED = summary_has_matching_offsets(score_summary, psite_offsets)
print(f'SCORES_ARE_REGISTERED = {SCORES_ARE_REGISTERED}')

if not scores_unique.empty and 'coverage_pass' in scores_unique and 'classes' in scores_unique:
    cds_mask = (
        scores_unique['sample_id'].eq(PRIMARY_SAMPLE)
        & scores_unique['coverage_pass'].astype(bool)
        & scores_unique['classes'].fillna('').astype(str).str.contains('cds')
    )
    cds_periodicity_median = scores_unique.loc[cds_mask, 'periodicity'].median()
    print(f'{PRIMARY_SAMPLE} coverage-pass CDS median periodicity: {cds_periodicity_median:.3f}')
    if not (SCORES_ARE_REGISTERED and cds_periodicity_median > 0.5):
        print('WARNING: registered periodicity check failed; periodicity evidence plots will be skipped.')
        SCORES_ARE_REGISTERED = False

save_table(scores_unique.head(0) if scores_unique.empty else scores_unique, 'scores_unique')
show_df('scores_unique', scores_unique)


## 6. Merge evidence table

`evidence_df` is the main table for score/evidence plots and example-locus ranking.


In [ ]:
if scores_unique.empty or agreement_df.empty:
    evidence_df = pd.DataFrame()
else:
    pooled_scores = scores_unique[scores_unique['sample_id'].eq(PRIMARY_SAMPLE)].copy()
    evidence_df = (
        pooled_scores
        .merge(agreement_df, on='feature_key', how='left', suffixes=('', '_agreement'))
        .merge(context_df, on='feature_key', how='left', suffixes=('', '_context'))
    )
    if 'classes' not in evidence_df and 'classes_context' in evidence_df:
        evidence_df['classes'] = evidence_df['classes_context']

save_table(evidence_df, 'evidence_df')
show_df('evidence_df', evidence_df)


## 7. Plot 1: Caller universe


In [ ]:
if calls_pooled.empty or context_df.empty:
    plot_caller_universe_df = pd.DataFrame()
    print('Skipping caller universe: pooled calls or context are unavailable.')
else:
    plot_caller_universe_df = (
        calls_pooled[['source_tool', 'feature_key']]
        .drop_duplicates()
        .merge(context_df[['feature_key', 'context_class']], on='feature_key', how='left')
        .groupby(['source_tool', 'context_class'])['feature_key']
        .nunique()
        .reset_index(name='n_candidates')
    )
    save_table(plot_caller_universe_df, '01_caller_universe')
    show_df('plot_caller_universe_df', plot_caller_universe_df, 20)

    pivot = plot_caller_universe_df.pivot_table(index='source_tool', columns='context_class', values='n_candidates', fill_value=0)
    pivot = pivot.reindex(index=[t for t in TOOL_ORDER if t in pivot.index], columns=[c for c in CONTEXT_ORDER if c in pivot.columns], fill_value=0)
    fig, ax = plt.subplots(figsize=(8.5, 4.8))
    left = np.zeros(len(pivot))
    y = np.arange(len(pivot))
    for cls in pivot.columns:
        vals = pivot[cls].values
        ax.barh(y, vals, left=left, color=CONTEXT_COLORS.get(cls, '0.5'), label=cls)
        left += vals
    ax.set_yticks(y)
    ax.set_yticklabels(pivot.index)
    ax.invert_yaxis()
    ax.set_xlabel('Unique pooled candidate ORFs')
    ax.set_title('Caller outputs differ in scale and composition')
    ax.grid(axis='x', alpha=0.25)
    ax.spines[['top', 'right']].set_visible(False)
    ax.legend(frameon=False, fontsize=8, loc='lower right')
    finish_plot(fig, '01_caller_universe.png')


## 8. Plot 2: Agreement geometry


In [ ]:
def agreement_summary_for_key(df, key_col, label):
    if df.empty:
        return pd.DataFrame(columns=['agreement_definition', 'support_tier', 'n_units', 'fraction'])
    counts = df.groupby(key_col)['source_tool'].nunique().reset_index(name='n_tools')
    counts['support_tier'] = counts['n_tools'].map(compact_support_tier)
    out = counts.groupby('support_tier').size().reset_index(name='n_units')
    out['agreement_definition'] = label
    out['fraction'] = out['n_units'] / out['n_units'].sum()
    return out[['agreement_definition', 'support_tier', 'n_units', 'fraction']]

plot_agreement_geometry_df = pd.concat([
    agreement_summary_for_key(calls_pooled, 'full_key', 'Full ORF'),
    agreement_summary_for_key(calls_pooled, 'start_key', 'Start'),
    agreement_summary_for_key(calls_pooled, 'stop_key', 'Stop'),
], ignore_index=True)
save_table(plot_agreement_geometry_df, '02_agreement_geometry')
show_df('plot_agreement_geometry_df', plot_agreement_geometry_df, 20)

if not plot_agreement_geometry_df.empty:
    order_x = ['Full ORF', 'Start', 'Stop']
    tier_order = ['1 tool', '2 tools', '3 tools', '4+ tools']
    pivot = plot_agreement_geometry_df.pivot_table(index='agreement_definition', columns='support_tier', values='fraction', fill_value=0).reindex(index=order_x, columns=tier_order, fill_value=0)
    colors = ['#c7c7c7', '#8ab6d6', '#4C72B0', '#1f3b73']
    fig, ax = plt.subplots(figsize=(7.0, 4.4))
    bottom = np.zeros(len(pivot))
    x = np.arange(len(pivot))
    for tier, color in zip(tier_order, colors):
        vals = pivot[tier].values
        ax.bar(x, vals, bottom=bottom, label=tier, color=color)
        bottom += vals
    ax.set_xticks(x)
    ax.set_xticklabels(pivot.index)
    ax.set_ylim(0, 1)
    ax.set_ylabel('Fraction of genomic units')
    ax.set_title('Agreement depends on which boundary is compared')
    ax.text(0.0, -0.20, 'Agreement is calculated in genomic space; transcript context is added afterward.', transform=ax.transAxes, fontsize=9, color='0.35')
    ax.legend(frameon=False, fontsize=8, loc='upper right')
    ax.grid(axis='y', alpha=0.25)
    ax.spines[['top', 'right']].set_visible(False)
    finish_plot(fig, '02_agreement_geometry.png')


## 9. Plot 3: Context of disagreement


In [ ]:
if agreement_df.empty or context_df.empty:
    plot_context_by_agreement_df = pd.DataFrame()
    print('Skipping context by agreement: inputs unavailable.')
else:
    plot_context_by_agreement_df = (
        agreement_df[['feature_key', 'agreement_class']]
        .merge(context_df[['feature_key', 'context_class']], on='feature_key', how='left')
        .groupby(['agreement_class', 'context_class'])['feature_key']
        .nunique()
        .reset_index(name='n_candidates')
    )
    plot_context_by_agreement_df['fraction'] = plot_context_by_agreement_df.groupby('agreement_class')['n_candidates'].transform(lambda s: s / s.sum())
    save_table(plot_context_by_agreement_df, '03_context_by_agreement')
    show_df('plot_context_by_agreement_df', plot_context_by_agreement_df, 20)

    row_order = [x for x in AGREEMENT_ORDER if x in set(plot_context_by_agreement_df['agreement_class'])]
    col_order = [x for x in CONTEXT_ORDER if x in set(plot_context_by_agreement_df['context_class'])]
    pivot = plot_context_by_agreement_df.pivot_table(index='agreement_class', columns='context_class', values='fraction', fill_value=0).reindex(index=row_order, columns=col_order, fill_value=0)
    fig, ax = plt.subplots(figsize=(8.2, 4.6))
    left = np.zeros(len(pivot))
    y = np.arange(len(pivot))
    for cls in pivot.columns:
        vals = pivot[cls].values
        ax.barh(y, vals, left=left, color=CONTEXT_COLORS.get(cls, '0.5'), label=cls)
        left += vals
    ax.set_yticks(y)
    ax.set_yticklabels([s.replace('_', ' ') for s in pivot.index])
    ax.invert_yaxis()
    ax.set_xlim(0, 1)
    ax.set_xlabel('Fraction of candidates')
    ax.set_title('Disagreement is concentrated in specific candidate contexts')
    ax.grid(axis='x', alpha=0.25)
    ax.spines[['top', 'right']].set_visible(False)
    ax.legend(frameon=False, fontsize=8, loc='lower right')
    finish_plot(fig, '03_context_by_agreement.png')


## 10. Plot 4: Start-choice ambiguity


In [ ]:
if calls_pooled.empty:
    plot_starts_per_stop_df = pd.DataFrame()
    print('Skipping starts-per-stop: pooled calls unavailable.')
else:
    stop_groups = (
        calls_pooled
        .groupby('stop_key')
        .agg(
            n_starts=('start_key', 'nunique'),
            n_tools_stop=('source_tool', 'nunique'),
            n_candidates=('feature_key', 'nunique'),
            min_start=('genomic_start', 'min'),
            max_start=('genomic_start', 'max'),
        )
        .reset_index()
    )
    stop_groups['start_span_nt'] = (stop_groups['max_start'] - stop_groups['min_start']).abs()
    stop_groups['n_starts_bucket'] = pd.cut(stop_groups['n_starts'], bins=[0, 1, 2, np.inf], labels=['1 start', '2 starts', '3+ starts'])
    stop_groups['shared_stop'] = np.where(stop_groups['n_tools_stop'].ge(2), 'shared by 2+ tools', 'single-tool stop')
    plot_starts_per_stop_df = stop_groups
    save_table(plot_starts_per_stop_df, '04_starts_per_stop')
    show_df('plot_starts_per_stop_df', plot_starts_per_stop_df, 20)

    summary = plot_starts_per_stop_df.groupby(['n_starts_bucket', 'shared_stop'], observed=False).size().reset_index(name='n_stop_groups')
    pivot = summary.pivot_table(index='n_starts_bucket', columns='shared_stop', values='n_stop_groups', fill_value=0)
    pivot = pivot.reindex(index=['1 start', '2 starts', '3+ starts'])
    fig, ax = plt.subplots(figsize=(6.8, 4.3))
    bottom = np.zeros(len(pivot))
    x = np.arange(len(pivot))
    for col, color in [('single-tool stop', '#c7c7c7'), ('shared by 2+ tools', '#4C72B0')]:
        vals = pivot[col].values if col in pivot else np.zeros(len(pivot))
        ax.bar(x, vals, bottom=bottom, label=col, color=color)
        bottom += vals
    ax.set_xticks(x)
    ax.set_xticklabels(pivot.index)
    ax.set_ylabel('Stop groups')
    ax.set_title('Many ORF disagreements are start-choice problems')
    ax.grid(axis='y', alpha=0.25)
    ax.spines[['top', 'right']].set_visible(False)
    ax.legend(frameon=False)
    finish_plot(fig, '04_starts_per_stop.png')


## 11. Plot 5: P-site calibration


In [ ]:
if plot_psite_offsets_df.empty:
    print('Skipping P-site calibration plot: offsets unavailable.')
else:
    plot_df = plot_psite_offsets_df.copy().sort_values('sample_id')
    plot_df['label'] = plot_df['sample_id'].replace({'Ribo_Pancreas_pooled': 'Pooled'}).str.replace('SRR1100', '', regex=False)
    fig, ax = plt.subplots(figsize=(7.4, 4.0))
    x = np.arange(len(plot_df))
    ax.scatter(x, plot_df['dom_frac'], s=76, color='#4C72B0', zorder=3)
    ax.axhline(1/3, color='0.65', linestyle=':', linewidth=1, label='random frame')
    ax.axhline(0.8, color='0.25', linestyle='--', linewidth=1, label='strong registration')
    if plot_df['offset'].nunique() == 1:
        ax.text(0.02, 0.93, f'all samples: offset = {int(plot_df["offset"].iloc[0])}', transform=ax.transAxes, ha='left', va='top', fontsize=10)
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df['label'], rotation=0)
    ax.set_ylim(0, 1.02)
    ax.set_ylabel('Dominant CDS-frame fraction')
    ax.set_title('P-site registration is consistent')
    ax.legend(frameon=False, loc='lower right')
    ax.grid(axis='y', alpha=0.25)
    ax.spines[['top', 'right']].set_visible(False)
    finish_plot(fig, '05_psite_offset_calibration.png')


## 12. Plot 6: Evidence by candidate class


In [ ]:
if not SCORES_ARE_REGISTERED or evidence_df.empty:
    plot_registered_periodicity_df = pd.DataFrame()
    print('Skipping registered periodicity by class: scores are not registered or evidence_df is empty.')
else:
    plot_registered_periodicity_df = evidence_df[evidence_df['coverage_pass'].astype(bool)].copy()
    plot_registered_periodicity_df['plot_group'] = np.where(plot_registered_periodicity_df['context_class'].eq('annotated_CDS'), 'Annotated CDS', 'Non-CDS candidates')
    save_table(plot_registered_periodicity_df[['feature_key', 'sample_id', 'plot_group', 'periodicity', 'coverage_pass', 'context_class']], '06_registered_periodicity_by_class')
    show_df('plot_registered_periodicity_df', plot_registered_periodicity_df)

    groups = ['Annotated CDS', 'Non-CDS candidates']
    data = [plot_registered_periodicity_df.loc[plot_registered_periodicity_df['plot_group'].eq(g), 'periodicity'].dropna() for g in groups]
    fig, ax = plt.subplots(figsize=(6.8, 4.4))
    parts = ax.violinplot(data, positions=[1, 2], widths=0.75, showmedians=True, showextrema=False)
    for body, color in zip(parts['bodies'], ['#4C72B0', '#DD8452']):
        body.set_facecolor(color)
        body.set_alpha(0.65)
    parts['cmedians'].set_color('black')
    parts['cmedians'].set_linewidth(2)
    for pos, vals in zip([1, 2], data):
        if len(vals):
            ax.text(pos, min(0.98, vals.median() + 0.04), f'median {vals.median():.2f}\nn={len(vals):,}', ha='center', va='bottom', fontsize=9)
    ax.axhline(1/3, color='0.45', linestyle=':', linewidth=1, label='random frame')
    ax.set_xticks([1, 2])
    ax.set_xticklabels(['Annotated CDS', 'Non-CDS\ncandidates'])
    ax.set_ylim(0, 1.02)
    ax.set_ylabel('Registered frame signal fraction')
    ax.set_title('P-site-corrected periodicity separates translated CDS')
    ax.legend(frameon=False, loc='upper right')
    ax.grid(axis='y', alpha=0.25)
    ax.spines[['top', 'right']].set_visible(False)
    finish_plot(fig, '06_registered_periodicity_by_class.png')


## 13. Plot 7: Evidence by agreement class


In [ ]:
if not SCORES_ARE_REGISTERED or evidence_df.empty:
    plot_evidence_by_agreement_df = pd.DataFrame()
    print('Skipping evidence by agreement: scores are not registered or evidence_df is empty.')
else:
    plot_evidence_by_agreement_df = evidence_df[evidence_df['coverage_pass'].astype(bool)].copy()
    plot_evidence_by_agreement_df = plot_evidence_by_agreement_df[plot_evidence_by_agreement_df['agreement_class'].notna()]
    save_table(plot_evidence_by_agreement_df[['feature_key', 'sample_id', 'agreement_class', 'periodicity', 'context_class']], '07_evidence_by_agreement')
    show_df('plot_evidence_by_agreement_df', plot_evidence_by_agreement_df)

    groups = [g for g in AGREEMENT_ORDER if g in set(plot_evidence_by_agreement_df['agreement_class'])]
    data = [plot_evidence_by_agreement_df.loc[plot_evidence_by_agreement_df['agreement_class'].eq(g), 'periodicity'].dropna() for g in groups]
    fig, ax = plt.subplots(figsize=(7.4, 4.4))
    parts = ax.violinplot(data, positions=np.arange(1, len(groups) + 1), widths=0.75, showmedians=True, showextrema=False)
    for body, group in zip(parts['bodies'], groups):
        body.set_facecolor(AGREEMENT_COLORS.get(group, '0.5'))
        body.set_alpha(0.65)
    parts['cmedians'].set_color('black')
    parts['cmedians'].set_linewidth(2)
    for pos, vals in zip(np.arange(1, len(groups) + 1), data):
        if len(vals):
            ax.text(pos, min(0.98, vals.median() + 0.04), f'{vals.median():.2f}\nn={len(vals):,}', ha='center', va='bottom', fontsize=9)
    ax.axhline(1/3, color='0.45', linestyle=':', linewidth=1, label='random frame')
    ax.set_xticks(np.arange(1, len(groups) + 1))
    ax.set_xticklabels([g.replace('_', ' ') for g in groups])
    ax.set_ylim(0, 1.02)
    ax.set_ylabel('Registered frame signal fraction')
    ax.set_title('Consensus is useful only if it tracks frame evidence')
    ax.legend(frameon=False, loc='upper right')
    ax.grid(axis='y', alpha=0.25)
    ax.spines[['top', 'right']].set_visible(False)
    finish_plot(fig, '07_evidence_by_agreement.png')


## 14. Plot 8: Coverage-pass by tool


In [ ]:
if scores_unique.empty:
    plot_coverage_pass_by_tool_df = pd.DataFrame()
    print('Skipping coverage-pass by tool: scores unavailable.')
else:
    scores_for_tool = scores_unique[scores_unique['sample_id'].eq(PRIMARY_SAMPLE)].copy()
    if 'source_tools' not in scores_for_tool:
        plot_coverage_pass_by_tool_df = pd.DataFrame()
        print('Skipping coverage-pass by tool: source_tools column missing.')
    else:
        bt = scores_for_tool.assign(tool=scores_for_tool['source_tools'].fillna('').str.split('|')).explode('tool')
        bt = bt[bt['tool'].isin(TOOL_ORDER)]
        plot_coverage_pass_by_tool_df = (
            bt.groupby('tool')
            .agg(fraction_coverage_pass=('coverage_pass', 'mean'), n_candidates=('feature_key', 'nunique'))
            .reset_index()
        )
        save_table(plot_coverage_pass_by_tool_df, '08_coverage_pass_by_tool')
        show_df('plot_coverage_pass_by_tool_df', plot_coverage_pass_by_tool_df, 20)

        plot_df = plot_coverage_pass_by_tool_df.set_index('tool').reindex([t for t in TOOL_ORDER if t in set(plot_coverage_pass_by_tool_df['tool'])]).reset_index()
        fig, ax = plt.subplots(figsize=(7.2, 4.2))
        bars = ax.bar(plot_df['tool'], plot_df['fraction_coverage_pass'], color=[TOOL_COLORS[t] for t in plot_df['tool']], width=0.68)
        for bar, row in zip(bars, plot_df.itertuples(index=False)):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.025, f'{row.fraction_coverage_pass:.0%}\nn={int(row.n_candidates):,}', ha='center', va='bottom', fontsize=9)
        ax.set_ylim(0, 1.08)
        ax.set_ylabel('Fraction with Ribo-seq coverage')
        ax.set_title('Most called ORFs have measurable Ribo-seq signal')
        ax.grid(axis='y', alpha=0.25)
        ax.spines[['top', 'right']].set_visible(False)
        finish_plot(fig, '08_coverage_pass_by_tool.png')


## 15. Ranked example locus tables

These tables are designed for hand-picking example loci. The top rows should be interpretable, not just extreme.


In [ ]:
example_loci_tables = {}

if evidence_df.empty:
    print('Skipping ranked loci: evidence_df is empty.')
else:
    rank_base = evidence_df.copy()
    rank_base['periodicity_rank_value'] = rank_base['periodicity'].fillna(-1)
    rank_base['body_total_signal_rank_value'] = rank_base['body_total_signal'].fillna(0)

    if not plot_starts_per_stop_df.empty:
        shared_stop = (
            rank_base
            .merge(plot_starts_per_stop_df[['stop_key', 'n_starts', 'n_tools_stop', 'n_candidates', 'start_span_nt']], on='stop_key', how='left')
            .query('n_starts >= 2 and n_tools_stop >= 2')
            .sort_values(['n_tools_stop', 'periodicity_rank_value', 'n_candidates'], ascending=[False, False, True])
        )
        example_loci_tables['loci_shared_stop_multiple_starts'] = shared_stop.head(50)

    tool_specific = (
        rank_base
        .query("agreement_class == 'tool_specific'")
        .query('coverage_pass == True')
        .query("context_class != 'annotated_CDS'")
        .sort_values(['periodicity_rank_value', 'body_total_signal_rank_value'], ascending=False)
    )
    example_loci_tables['loci_tool_specific_high_evidence'] = tool_specific.head(50)

    consensus_low = (
        rank_base
        .query('n_tools_full >= 2')
        .query('coverage_pass == True')
        .sort_values(['n_tools_full', 'periodicity_rank_value'], ascending=[False, True])
    )
    example_loci_tables['loci_full_consensus_low_evidence'] = consensus_low.head(50)

    # Simple overlap clusters: interval overlap within chrom/strand among pooled candidates.
    cluster_rows = []
    cluster_id = 0
    if not agreement_df.empty:
        for (chrom, strand), grp in agreement_df.sort_values(['bed_chrom', 'bed_strand', 'bed_start', 'bed_end']).groupby(['bed_chrom', 'bed_strand']):
            active = []
            current = []
            current_end = None
            for row in grp.itertuples(index=False):
                start, end = int(row.bed_start), int(row.bed_end)
                if current and start > current_end:
                    cluster_id += 1
                    for r in current:
                        cluster_rows.append({'feature_key': r.feature_key, 'overlap_cluster_id': cluster_id, 'cluster_size': len(current)})
                    current = []
                    current_end = None
                current.append(row)
                current_end = end if current_end is None else max(current_end, end)
            if current:
                cluster_id += 1
                for r in current:
                    cluster_rows.append({'feature_key': r.feature_key, 'overlap_cluster_id': cluster_id, 'cluster_size': len(current)})
    overlap_clusters = pd.DataFrame(cluster_rows)
    if not overlap_clusters.empty:
        complex_overlap = (
            rank_base
            .merge(overlap_clusters, on='feature_key', how='left')
            .query('cluster_size >= 3')
            .sort_values(['cluster_size', 'n_tools_full', 'periodicity_rank_value'], ascending=[False, False, False])
        )
    else:
        complex_overlap = pd.DataFrame()
    example_loci_tables['loci_complex_overlap_candidates'] = complex_overlap.head(50)

for name, df in example_loci_tables.items():
    save_table(df, name)
    print(f'{name}: {df.shape}')
    display(df.head(5))


## 16. Simple locus plots

The locus plot is intentionally minimal: candidate intervals plus optional coverage if a `BigWigPair` is supplied later.


In [ ]:
def plot_locus(locus_row, candidates_df, scores_df=None, bws=None, window_nt=150, out=None):
    if isinstance(locus_row, pd.Series):
        row = locus_row
    else:
        row = pd.Series(locus_row)

    chrom = row['bed_chrom']
    strand = row['bed_strand']
    center_start = int(row['bed_start'])
    center_end = int(row['bed_end'])
    start = max(0, center_start - window_nt)
    end = center_end + window_nt

    locus_candidates = candidates_df[
        candidates_df['bed_chrom'].eq(chrom)
        & candidates_df['bed_strand'].eq(strand)
        & candidates_df['bed_end'].ge(start)
        & candidates_df['bed_start'].le(end)
    ].copy()
    if locus_candidates.empty:
        print('No candidates in locus window.')
        return None

    fig, axes = plt.subplots(2, 1, figsize=(9.5, 3.8), sharex=True, gridspec_kw={'height_ratios': [1.8, 1]})
    ax = axes[0]
    locus_candidates = locus_candidates.sort_values(['bed_start', 'bed_end']).reset_index(drop=True)
    for y, cand in enumerate(locus_candidates.itertuples(index=False)):
        color = AGREEMENT_COLORS.get(getattr(cand, 'agreement_class', 'tool_specific'), '0.5')
        ax.plot([cand.bed_start, cand.bed_end], [y, y], color=color, lw=7, solid_capstyle='butt')
        label = f"{getattr(cand, 'n_tools_full', '?')} tools | {getattr(cand, 'context_class', '')}"
        ax.text(cand.bed_end + max(1, (end - start) * 0.01), y, label, va='center', fontsize=8)
    ax.axvspan(center_start, center_end, color='0.9', zorder=-1)
    ax.set_yticks([])
    ax.set_ylabel('candidate ORFs')
    ax.set_title(f'{chrom}:{start:,}-{end:,} ({strand})')
    ax.spines[['top', 'right', 'left']].set_visible(False)

    cov_ax = axes[1]
    if bws is not None:
        try:
            vals = bws.values(chrom, start, end, strand, reverse=False)
            if vals is not None:
                xs = np.arange(start, start + len(vals))
                cov_ax.plot(xs, vals, color='black', lw=0.8)
                cov_ax.set_ylabel('P-site signal')
            else:
                cov_ax.text(0.5, 0.5, 'no bigWig signal', transform=cov_ax.transAxes, ha='center', va='center')
        except Exception as exc:
            cov_ax.text(0.5, 0.5, f'coverage unavailable: {exc}', transform=cov_ax.transAxes, ha='center', va='center')
    else:
        cov_ax.text(0.5, 0.5, 'coverage track not loaded', transform=cov_ax.transAxes, ha='center', va='center')
    cov_ax.set_xlabel('genomic coordinate')
    cov_ax.spines[['top', 'right']].set_visible(False)

    fig.tight_layout()
    if out is not None:
        fig.savefig(out, dpi=180, bbox_inches='tight')
        print(f'Saved {out}')
    plt.show()
    return fig

# Enrich candidates for locus plotting.
locus_candidates_df = agreement_df.merge(context_df[['feature_key', 'context_class']], on='feature_key', how='left') if not agreement_df.empty and not context_df.empty else agreement_df

example_specs = [
    ('loci_shared_stop_multiple_starts', '09_locus_shared_stop_multiple_starts.png'),
    ('loci_tool_specific_high_evidence', '10_locus_tool_specific_high_evidence.png'),
    ('loci_full_consensus_low_evidence', '11_locus_consensus_low_evidence.png'),
]
for table_name, png_name in example_specs:
    table = example_loci_tables.get(table_name, pd.DataFrame())
    if table.empty:
        print(f'Skipping {png_name}: {table_name} is empty')
        continue
    plot_locus(table.iloc[0], locus_candidates_df, scores_unique, bws=None, out=OUT_DIR / png_name)


## 17. Final readiness checklist

Before using periodicity plots in slides, verify:
- `SCORES_ARE_REGISTERED` is `True`.
- `signal_score_summary.json` records `psite_offset` for each sample.
- `Ribo_Pancreas_pooled` coverage-pass annotated CDS median periodicity is above 0.5.
- The plot dataframe TSV for each slide exists in `tables_talk_foundation/`.
